# Dental Detection Notebook


In [ ]:
import sys
!{sys.executable} -m pip install -r requirements.txt

## Imports


In [ ]:
import json
import os
import random
import tempfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
import torchvision
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch.utils.data import DataLoader
from torchvision import tv_tensors
from torchvision.datasets import CocoDetection
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.rpn import AnchorGenerator, RPNHead
from torchvision.transforms import v2 as T

print("torch", torch.__version__)
print("torchvision", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())


## Configuration


In [ ]:
ROOT = Path.cwd()
DATASET_ROOT = ROOT / "Dentex-licenta-dataset"

TRAIN_IMAGES = DATASET_ROOT / "training_data" / "quadrant-enumeration-disease" / "xrays"
TRAIN_ANN = DATASET_ROOT / "training_data" / "quadrant-enumeration-disease" / "train_quadrant_enumeration_disease.json"

VAL_IMAGES = DATASET_ROOT / "validation_data" / "quadrant_enumeration_disease" / "xrays"
VAL_ANN = DATASET_ROOT / "validation_data" / "quadrant_enumeration_disease" / "validation_triple_updated.json"
if not VAL_ANN.exists():
    VAL_ANN = DATASET_ROOT / "validation_data" / "quadrant_enumeration_disease" / "validation_triple.json"

TEST_IMAGES = DATASET_ROOT / "test_data" / "xrays"
TEST_ANN = DATASET_ROOT / "test_data" / "quadrant-enumeration-disease" / "test_quadrant_enumeration_disease.json"

OUTPUT_PATH = ROOT / "notebook_best_fasterrcnn.pth"

BATCH_SIZE = 2
EPOCHS = 20
ACCUMULATION_STEPS = 4
NUM_WORKERS = 0
FREEZE_EPOCHS = 4
WARMUP_EPOCHS = 2
EARLY_STOP = 8
IMAGE_MIN_SIZE = 1200
IMAGE_MAX_SIZE = 2200
HEAD_LR = 2e-4
BACKBONE_LR = 1e-5
FINETUNE_HEAD_LR = 5e-5
WEIGHT_DECAY = 5e-4
GRAD_CLIP = 0.5
SEED = 42

for path in [DATASET_ROOT, TRAIN_IMAGES, TRAIN_ANN, VAL_IMAGES, VAL_ANN, TEST_IMAGES, TEST_ANN]:
    print(path, "OK" if path.exists() else "MISSING")


## Data Preparation


In [ ]:
def normalize_dentex_json(input_path, output_path=None):
    input_path = Path(input_path)
    data = json.loads(input_path.read_text())

    categories_3 = data.get("categories_3")
    if not categories_3:
        raise ValueError(f"{input_path} is missing categories_3")

    raw_ids = [int(cat["id"]) for cat in categories_3]
    id_offset = 1 if raw_ids and min(raw_ids) == 0 else 0
    id_mapping = {raw_id: raw_id + id_offset for raw_id in raw_ids}

    categories = []
    for cat in categories_3:
        raw_id = int(cat["id"])
        categories.append(
            {
                "id": id_mapping[raw_id],
                "name": cat["name"],
                "supercategory": cat.get("supercategory", cat["name"]),
            }
        )

    annotations = []
    next_ann_id = 1
    for ann in data.get("annotations", []):
        if "category_id_3" not in ann:
            continue
        x, y, w, h = ann["bbox"]
        if w <= 1 or h <= 1:
            continue
        raw_category_id = int(ann["category_id_3"])
        annotations.append(
            {
                "id": int(ann.get("id", next_ann_id)),
                "image_id": int(ann["image_id"]),
                "category_id": id_mapping.get(raw_category_id, raw_category_id + id_offset),
                "bbox": [float(x), float(y), float(w), float(h)],
                "area": float(ann.get("area", w * h)),
                "iscrowd": int(ann.get("iscrowd", 0)),
            }
        )
        next_ann_id += 1

    normalized = {
        "images": data.get("images", []),
        "annotations": annotations,
        "categories": sorted(categories, key=lambda item: item["id"]),
    }

    if output_path is None:
        output_path = input_path.with_name(input_path.stem + "_coco.json")
    output_path = Path(output_path)
    output_path.write_text(json.dumps(normalized))
    return output_path


def summarize_dataset(name, ann_path):
    ann_path = Path(ann_path)
    data = json.loads(ann_path.read_text())
    counts = Counter(int(ann["category_id"]) for ann in data["annotations"])
    cat_names = {int(cat["id"]): cat["name"] for cat in data["categories"]}
    readable = {cat_names[k]: counts.get(k, 0) for k in sorted(cat_names)}
    missing = [cat_names[k] for k in sorted(cat_names) if counts.get(k, 0) == 0]
    print(f"{name}: {len(data['images'])} images | {len(data['annotations'])} boxes | {readable}")
    if missing:
        print("  Missing classes:", ", ".join(missing))
    return cat_names


TRAIN_ANN_COCO = normalize_dentex_json(TRAIN_ANN)
VAL_ANN_COCO = normalize_dentex_json(VAL_ANN)
TEST_ANN_COCO = normalize_dentex_json(TEST_ANN)

CAT_NAMES = summarize_dataset("Train", TRAIN_ANN_COCO)
summarize_dataset("Validation", VAL_ANN_COCO)
summarize_dataset("Test", TEST_ANN_COCO)

NUM_CLASSES = max(CAT_NAMES) + 1
CAT_NAMES


In [ ]:
def draw_annotations(image, boxes, labels, title):
    if isinstance(image, torch.Tensor):
        image_np = image.detach().cpu().permute(1, 2, 0).numpy()
    else:
        image_np = np.array(image)
    image_np = np.clip(image_np, 0, 1) if image_np.dtype != np.uint8 else image_np

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(image_np)
    for box, label in zip(boxes, labels):
        x1, y1, x2, y2 = [float(v) for v in box]
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="red", facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1, CAT_NAMES[int(label)], color="yellow", fontsize=9, backgroundcolor="black")
    ax.set_title(title)
    ax.axis("off")
    plt.show()


In [ ]:
class DentalCocoDataset(CocoDetection):
    def __init__(self, image_root, ann_file, transforms=None):
        super().__init__(str(image_root), str(ann_file))
        self.sample_transforms = transforms
        self.transforms = None

    @staticmethod
    def get_canvas_size(img):
        if hasattr(img, "size") and isinstance(img.size, tuple) and len(img.size) == 2:
            width, height = img.size
            return int(height), int(width)
        if hasattr(img, "shape") and len(img.shape) >= 2:
            if len(img.shape) == 2:
                height, width = img.shape
            else:
                height, width = img.shape[-2], img.shape[-1]
            return int(height), int(width)
        raise TypeError(f"Cannot infer size for {type(img)!r}")

    def __getitem__(self, idx):
        img, anns = super().__getitem__(idx)
        canvas_size = self.get_canvas_size(img)

        boxes = []
        labels = []
        areas = []
        iscrowd = []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w <= 1 or h <= 1:
                continue
            boxes.append([x, y, x + w, y + h])
            labels.append(int(ann["category_id"]))
            areas.append(float(ann.get("area", w * h)))
            iscrowd.append(int(ann.get("iscrowd", 0)))

        if boxes:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            areas = torch.tensor(areas, dtype=torch.float32)
            iscrowd = torch.tensor(iscrowd, dtype=torch.int64)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            areas = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=canvas_size),
            "labels": labels,
            "image_id": torch.tensor([self.ids[idx]], dtype=torch.int64),
            "area": areas,
            "iscrowd": iscrowd,
        }

        if self.sample_transforms is not None:
            img, target = self.sample_transforms(img, target)

        target["boxes"] = torch.as_tensor(target["boxes"], dtype=torch.float32)
        target["labels"] = target["labels"].to(dtype=torch.int64)
        return img, target


def build_transforms(train):
    transforms = [T.ToImage(), T.ToDtype(torch.float32, scale=True)]
    if train:
        transforms.extend(
            [
                T.RandomHorizontalFlip(0.5),
                T.RandomAffine(degrees=(-5, 5), translate=(0.03, 0.03), scale=(0.95, 1.05), fill=0),
                T.RandomPhotometricDistort(p=0.25),
                T.ClampBoundingBoxes(),
                T.SanitizeBoundingBoxes(min_size=2),
            ]
        )
    return T.Compose(transforms)


def collate_fn(batch):
    return tuple(zip(*batch))


train_preview = DentalCocoDataset(TRAIN_IMAGES, TRAIN_ANN_COCO, transforms=build_transforms(train=False))
preview_image, preview_target = train_preview[0]
draw_annotations(preview_image, preview_target["boxes"], preview_target["labels"], "Training sample")


## Model Training


In [ ]:
def build_model(num_classes, min_size=IMAGE_MIN_SIZE, max_size=IMAGE_MAX_SIZE):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
        weights="DEFAULT",
        min_size=min_size,
        max_size=max_size,
    )

    anchor_generator = AnchorGenerator(
        sizes=((32, 48), (64, 96), (128, 160), (192, 256), (320, 384)),
        aspect_ratios=((0.5, 0.75, 1.0, 1.5, 2.0),) * 5,
    )
    model.rpn.anchor_generator = anchor_generator
    model.rpn.head = RPNHead(model.backbone.out_channels, anchor_generator.num_anchors_per_location()[0])
    model.rpn.nms_thresh = 0.75
    model.rpn.post_nms_top_n_train = 2000
    model.rpn.post_nms_top_n_test = 1500
    model.roi_heads.score_thresh = 0.001
    model.roi_heads.nms_thresh = 0.5
    model.roi_heads.detections_per_img = 300

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


@torch.no_grad()
def evaluate_map50(model, data_loader, device, coco_gt):
    model.eval()
    results = []

    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        for target, output in zip(targets, outputs):
            image_id = int(target["image_id"].item())
            boxes = output["boxes"].detach().cpu()
            scores = output["scores"].detach().cpu()
            labels = output["labels"].detach().cpu()

            for box, score, label in zip(boxes, scores, labels):
                x1, y1, x2, y2 = box.tolist()
                results.append(
                    {
                        "image_id": image_id,
                        "category_id": int(label),
                        "bbox": [x1, y1, x2 - x1, y2 - y1],
                        "score": float(score),
                    }
                )

    if not results:
        return 0.0, None

    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as handle:
        json.dump(results, handle)
        result_path = handle.name

    try:
        coco_dt = coco_gt.loadRes(result_path)
        coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()
        return float(coco_eval.stats[1]), coco_eval
    finally:
        os.remove(result_path)


def print_per_class_ap50(coco_eval, coco_gt):
    if coco_eval is None:
        print("No detections available for AP50 breakdown.")
        return
    precisions = coco_eval.eval["precision"]
    for cat_index, cat_id in enumerate(coco_eval.params.catIds):
        values = precisions[0, :, cat_index, 0, -1]
        values = values[values > -1]
        name = coco_gt.cats[cat_id]["name"]
        if values.size:
            print(f"AP50 {name}: {float(values.mean()):.4f}")
        else:
            print(f"AP50 {name}: n/a")


In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

train_dataset = DentalCocoDataset(TRAIN_IMAGES, TRAIN_ANN_COCO, transforms=build_transforms(train=True))
val_dataset = DentalCocoDataset(VAL_IMAGES, VAL_ANN_COCO, transforms=build_transforms(train=False))
test_dataset = DentalCocoDataset(TEST_IMAGES, TEST_ANN_COCO, transforms=build_transforms(train=False))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)

coco_val = COCO(str(VAL_ANN_COCO))
coco_test = COCO(str(TEST_ANN_COCO))

print("train size:", len(train_dataset))
print("validation size:", len(val_dataset))
print("test size:", len(test_dataset))


In [ ]:
def train_model():
    model = build_model(NUM_CLASSES).to(device)

    for name, param in model.named_parameters():
        if "backbone" in name:
            param.requires_grad = False

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
    warmup = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=0.1,
        end_factor=1.0,
        total_iters=max(WARMUP_EPOCHS, 1),
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=4,
        min_lr=1e-7,
    )

    best_map50 = 0.0
    epochs_without_improvement = 0
    history = []

    for epoch in range(EPOCHS):
        if epoch == FREEZE_EPOCHS:
            print("Unfreezing backbone")
            for param in model.parameters():
                param.requires_grad = True

            backbone_params = [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad]
            head_params = [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad]
            optimizer = torch.optim.AdamW(
                [
                    {"params": backbone_params, "lr": BACKBONE_LR},
                    {"params": head_params, "lr": FINETUNE_HEAD_LR},
                ],
                weight_decay=WEIGHT_DECAY,
            )
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=max(EPOCHS - FREEZE_EPOCHS, 1),
                eta_min=1e-8,
            )

        model.train()
        optimizer.zero_grad()
        running_loss = 0.0

        for batch_idx, (images, targets) in enumerate(train_loader):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in target.items()} for target in targets]

            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
            (loss / ACCUMULATION_STEPS).backward()

            boundary = (batch_idx + 1) % ACCUMULATION_STEPS == 0
            last_batch = (batch_idx + 1) == len(train_loader)
            if boundary or last_batch:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
                optimizer.step()
                optimizer.zero_grad()

            running_loss += float(loss.item())

        map50, coco_eval = evaluate_map50(model, val_loader, device, coco_val)
        avg_loss = running_loss / max(len(train_loader), 1)
        lr = optimizer.param_groups[0]["lr"]
        history.append({"epoch": epoch + 1, "loss": avg_loss, "map50": map50, "lr": lr})
        print(f"Epoch {epoch + 1:03d}/{EPOCHS} | loss={avg_loss:.4f} | mAP50={map50:.4f} | lr={lr:.8f}")

        if epoch < WARMUP_EPOCHS and epoch < FREEZE_EPOCHS:
            warmup.step()
        elif epoch < FREEZE_EPOCHS:
            scheduler.step(map50)
        else:
            scheduler.step()

        if map50 > best_map50:
            best_map50 = map50
            epochs_without_improvement = 0
            torch.save(model.state_dict(), OUTPUT_PATH)
            print(f"Saved best model to {OUTPUT_PATH}")
            print_per_class_ap50(coco_eval, coco_val)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOP:
                print("Early stopping triggered")
                break

    return model, history


In [ ]:
model, history = train_model()
history[-5:] if history else history


## Validation And Test Inspection


In [ ]:
best_model = build_model(NUM_CLASSES).to(device)
best_model.load_state_dict(torch.load(OUTPUT_PATH, map_location=device))
best_model.eval()

test_map50, test_eval = evaluate_map50(best_model, test_loader, device, coco_test)
print("Test mAP50:", test_map50)
print_per_class_ap50(test_eval, coco_test)


In [ ]:
def predict_single(model, dataset, index=0, score_threshold=0.35):
    image, target = dataset[index]
    with torch.no_grad():
        output = model([image.to(device)])[0]

    keep = output["scores"].detach().cpu() >= score_threshold
    pred_boxes = output["boxes"].detach().cpu()[keep]
    pred_labels = output["labels"].detach().cpu()[keep]
    pred_scores = output["scores"].detach().cpu()[keep]

    print("Ground truth boxes:", len(target["boxes"]))
    print("Predicted boxes:", len(pred_boxes))
    if len(pred_scores):
        print("Scores:", [round(float(s), 3) for s in pred_scores[:10]])

    draw_annotations(image, target["boxes"], target["labels"], f"Ground truth sample #{index}")
    draw_annotations(image, pred_boxes, pred_labels, f"Predictions sample #{index}")


predict_single(best_model, test_dataset, index=0, score_threshold=0.35)
